# Classification de bâtiments — cas de test et résultats

Notre projet classe une image dans une des trois catégories suivantes :

| Label | Classe |
|---:|---|
| 0 | Art déco |
| 1 | Art nouveau |
| 2 | Gothique |

Les quatre algorithmes sont écrits en C++ dans `ml_library` : Perceptron, MLP, RBF et SVM. Python sert seulement à lire les résultats, dessiner les graphiques et lancer les tests C++.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

import matplotlib.pyplot as plt
import numpy as np

racine = Path.cwd().resolve()
while racine != racine.parent and not (racine / ".git").exists():
    racine = racine.parent

# Racine Git : meme chemin sur nos PC.
projet = (
    racine
    / "notebook_modeles_entraines_batiments_clean"
    / "app_final"
    / "projet_rendu_2_ml"
)
dossier_build = racine / "cmake-build-debug"
dossier_modeles = projet / "models"
dossier_rapports = dossier_modeles / "reports"
chemin_manifeste = dossier_modeles / "models_manifest.json"

noms_classes = ["Art déco", "Art nouveau", "Gothique"]

print("Projet :", projet)
print("Manifest :", chemin_manifeste)

## 1. Dataset et prétraitement

Le dataset sans doublon contient **1 852 images** :

- Art déco : 554 images ;
- Art nouveau : 745 images ;
- Gothique : 553 images.

Le fichier utilisé pour l'entraînement est `data/batiments_3_classes.csv`. Chaque image est corrigée selon son orientation, convertie en niveaux de gris, redimensionnée en 32 × 32, normalisée entre 0 et 1 puis aplatie en **1 024 valeurs**. Le label est ajouté à la fin de la ligne.

Le même prétraitement est utilisé pour créer le CSV et pour préparer une image envoyée à Flask. Ce prétraitement n'est pas du machine learning.

In [ ]:
nombre_images = [554, 745, 553]
couleurs = ["#5B8FF9", "#61DDAA", "#65789B"]

figure, axe = plt.subplots(figsize=(7, 4))
axe.bar(noms_classes, nombre_images, color=couleurs)
axe.set_title("Répartition des 1 852 images")
axe.set_ylabel("Nombre d'images")

for position in range(len(nombre_images)):
    axe.text(position, nombre_images[position] + 10, nombre_images[position], ha="center")

plt.tight_layout()
plt.show()

## 2. Architecture générale

L'entraînement est fait avant l'utilisation de l'application web :

```text
CSV → train_cli.exe → ml_api.cpp → ml_library → modèle sauvegardé
```

La prédiction utilise ensuite ce modèle déjà entraîné :

```text
Navigateur → Flask → predict_cli.exe → ml_load → ml_predict → résultat
```

Flask ne réentraîne jamais un modèle. Il prépare l'image, appelle `predict_cli.exe` et affiche le JSON retourné par le C++.

## 3. Cas de test synthétiques

Ces petits jeux de données servent à comprendre les limites des modèles. Ils sont seulement dessinés avec Python. Les vrais entraînements de test seront lancés ensuite par `demo.exe --tests`.

On retrouve les cas classiques du notebook fourni par le professeur : données linéaires, XOR, croix et classification à trois classes.

In [ ]:
generateur = np.random.default_rng(42)

def dessiner_cas(axe, donnees, etiquettes, titre):
    couleurs_points = np.array(["#5B8FF9", "#F6BD16", "#E8684A"])
    axe.scatter(
        donnees[:, 0],
        donnees[:, 1],
        c=couleurs_points[etiquettes],
        s=18
    )
    axe.set_title(titre)
    axe.grid(alpha=0.2)

donnees_lineaires = np.array([
    [1, 1], [1.5, 2], [2, 1], [2, 2.5],
    [5, 5], [6, 5], [5.5, 6], [6, 6.5]
])
etiquettes_lineaires = np.array([0, 0, 0, 0, 1, 1, 1, 1])

groupe_1 = generateur.normal([1, 1], 0.25, (60, 2))
groupe_2 = generateur.normal([2.2, 2.2], 0.25, (60, 2))
donnees_lineaires_multiples = np.vstack([groupe_1, groupe_2])
etiquettes_lineaires_multiples = np.array([0] * 60 + [1] * 60)

donnees_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
etiquettes_xor = np.array([0, 1, 1, 0])

donnees_croix = generateur.uniform(-1, 1, (500, 2))
dans_barre_verticale = np.abs(donnees_croix[:, 0]) < 0.25
dans_barre_horizontale = np.abs(donnees_croix[:, 1]) < 0.25
etiquettes_croix = (dans_barre_verticale | dans_barre_horizontale).astype(int)

classe_0 = generateur.normal([-1, -0.5], 0.25, (80, 2))
classe_1 = generateur.normal([1, -0.4], 0.25, (80, 2))
classe_2 = generateur.normal([0, 1], 0.25, (80, 2))
donnees_trois_classes = np.vstack([classe_0, classe_1, classe_2])
etiquettes_trois_classes = np.array([0] * 80 + [1] * 80 + [2] * 80)

donnees_trois_zones = generateur.uniform(-1, 1, (700, 2))
distance_du_centre = np.sqrt(
    donnees_trois_zones[:, 0] ** 2 + donnees_trois_zones[:, 1] ** 2
)
etiquettes_trois_zones = np.full(700, 2)
etiquettes_trois_zones[distance_du_centre < 0.35] = 0
zone_droite = donnees_trois_zones[:, 0] > donnees_trois_zones[:, 1]
etiquettes_trois_zones[(distance_du_centre >= 0.35) & zone_droite] = 1

liste_cas = [
    (donnees_lineaires, etiquettes_lineaires, "Linéaire simple"),
    (donnees_lineaires_multiples, etiquettes_lineaires_multiples, "Linéaire multiple"),
    (donnees_xor, etiquettes_xor, "XOR"),
    (donnees_croix, etiquettes_croix, "Croix"),
    (donnees_trois_classes, etiquettes_trois_classes, "Trois classes"),
    (donnees_trois_zones, etiquettes_trois_zones, "Trois zones")
]

figure, axes = plt.subplots(2, 3, figsize=(14, 8))
for position in range(len(liste_cas)):
    donnees, etiquettes, titre = liste_cas[position]
    dessiner_cas(axes.flat[position], donnees, etiquettes, titre)

plt.tight_layout()
plt.show()

## 4. Exécution des vrais tests C++

`demo.exe --tests` entraîne les quatre algorithmes sur trois petits cas : données linéaires, XOR et trois classes. Il sauvegarde aussi chaque modèle, le recharge et vérifie que le résultat reste identique.

Ces tests utilisent de petits points d'entraînement puis des points de test jamais vus. Ils vérifient le comportement du code et la sauvegarde, mais ne remplacent pas le test final sur les 370 images de bâtiments.

In [ ]:
# Build CLion : cree a la racine du depot.
chemin_demo = dossier_build / "demo.exe"
chemin_mingw = Path(
    r"C:\Program Files\JetBrains\CLion 2025.2.2\bin\mingw\bin"
)

environnement = os.environ.copy()
ancien_path = environnement.get("PATH", "")
environnement["PATH"] = str(chemin_mingw) + os.pathsep + ancien_path

if chemin_demo.exists():
    resultat_test = subprocess.run(
        [str(chemin_demo), "--tests"],
        cwd=projet,
        env=environnement,
        text=True,
        capture_output=True,
        timeout=60
    )
    print(resultat_test.stdout)
    if resultat_test.stderr:
        print(resultat_test.stderr)
    print("Code retour :", resultat_test.returncode)
else:
    print("demo.exe est absent. Il faut compiler la cible demo dans CLion.")

### Lecture du test XOR

Le Perceptron et le SVM linéaire restent limités sur XOR, car une seule frontière droite ne suffit pas. Le PMC et le RBF peuvent apprendre ce cas non linéaire. `demo.exe --tests` reste le point d'entrée unique pour ces tests pédagogiques.

## 5. Résultats sur les bâtiments

Les cellules suivantes lisent `models_manifest.json` et les rapports déjà enregistrés. Elles ne lancent aucun entraînement. Le manifest contient les quatre modèles proposés dans l'interface web.

In [ ]:
def lire_json(chemin):
    texte = chemin.read_text(encoding="utf-8")
    return json.loads(texte)

def afficher_pourcentage(valeur):
    if valeur is None:
        return "n/a"
    return f"{valeur * 100:.2f} %"

if chemin_manifeste.exists():
    manifeste = lire_json(chemin_manifeste)
    liste_modeles = manifeste.get("models", [])
else:
    liste_modeles = []
    print("Manifest absent :", chemin_manifeste)

print(f"{'Modèle':<18} {'Train':<12} {'Test':<12} Fichier")
print("-" * 80)

for modele in liste_modeles:
    chemin_modele = dossier_modeles / modele["file"]
    etat = "OK" if chemin_modele.exists() else "MANQUANT"
    train = afficher_pourcentage(modele.get("train_accuracy"))
    test = afficher_pourcentage(modele.get("test_accuracy"))
    print(f"{modele['id']:<18} {train:<12} {test:<12} {chemin_modele.name} [{etat}]")

In [ ]:
noms_modeles = []
accuracies_test = []
accuracies_equilibrees = []

for modele in liste_modeles:
    noms_modeles.append(modele["id"])
    accuracies_test.append(modele["test_accuracy"])

    accuracy_equilibree = modele.get("balanced_accuracy")
    if accuracy_equilibree is None:
        matrice = np.array(modele["confusion_matrix"])
        rappels = []
        for numero_classe in range(3):
            total_classe = matrice[numero_classe].sum()
            bonnes_reponses = matrice[numero_classe, numero_classe]
            rappels.append(bonnes_reponses / total_classe)
        accuracy_equilibree = sum(rappels) / len(rappels)

    accuracies_equilibrees.append(accuracy_equilibree)

if liste_modeles:
    positions = np.arange(len(noms_modeles))
    largeur = 0.35

    figure, axe = plt.subplots(figsize=(9, 4))
    axe.bar(positions - largeur / 2, accuracies_test, largeur, label="Accuracy test")
    axe.bar(positions + largeur / 2, accuracies_equilibrees, largeur, label="Balanced accuracy")
    axe.set_xticks(positions, noms_modeles, rotation=15)
    axe.set_ylim(0, 1)
    axe.set_title("Résultats des modèles du manifest")
    axe.legend()
    plt.tight_layout()
    plt.show()

## 6. Matrices de confusion

Une ligne correspond à la vraie classe et une colonne à la classe prédite. Les bonnes prédictions sont sur la diagonale. Les autres cases montrent les confusions entre styles architecturaux.

In [ ]:
def dessiner_matrice(axe, matrice, titre):
    matrice = np.array(matrice, dtype=int)
    axe.imshow(matrice, cmap="Blues")
    axe.set_title(titre)
    axe.set_xlabel("Classe prédite")
    axe.set_ylabel("Classe réelle")
    axe.set_xticks(range(3), noms_classes, rotation=30, ha="right")
    axe.set_yticks(range(3), noms_classes)

    for ligne in range(3):
        for colonne in range(3):
            valeur = matrice[ligne, colonne]
            axe.text(colonne, ligne, valeur, ha="center", va="center")

if liste_modeles:
    figure, axes = plt.subplots(2, 2, figsize=(11, 9))
    for position in range(min(4, len(liste_modeles))):
        modele = liste_modeles[position]
        dessiner_matrice(
            axes.flat[position],
            modele["confusion_matrix"],
            modele["id"]
        )
    plt.tight_layout()
    plt.show()

## 7. Analyse de MLP et RBF

**MLP v2** est le modèle conseillé pour la démonstration web. Il obtient la meilleure accuracy test du manifest. Il utilise 32 neurones cachés, 12 epochs et un learning rate de 0,02.

**RBF v1** prédisait Gothique pour tous les exemples du test. En 1 024 dimensions, son sigma de 1 rendait presque toutes les activations gaussiennes nulles. Le biais de sortie finissait alors par dominer.

**RBF v2** utilise 96 centres mieux répartis et un sigma de 5. Il ne prédit plus une seule classe partout et obtient environ 51,89 % sur le test. Il reste moins performant que MLP v2, mais la correction est réelle.

In [ ]:
versions_a_comparer = ["mlp_v1", "mlp_v2", "rbf_v1", "rbf_v2"]

for identifiant in versions_a_comparer:
    chemin_rapport = dossier_rapports / f"{identifiant}.json"
    if not chemin_rapport.exists():
        print(identifiant, ": rapport absent")
        continue

    rapport = lire_json(chemin_rapport)
    matrice = np.array(rapport["confusion_matrix"])
    repartition_predictions = matrice.sum(axis=0).tolist()

    print("\n", identifiant)
    print("  accuracy test :", afficher_pourcentage(rapport["test_accuracy"]))
    print("  paramètres :", rapport["parameters"])
    print("  prédictions [0, 1, 2] :", repartition_predictions)

## 8. Commandes utiles pour la soutenance

Les commandes sont lancées depuis `projet_rendu_2_ml`.

### Cas de test

```powershell
.\cmake-build-debug\demo.exe --tests
```

### Entraîner un modèle sans remplacer les modèles officiels

```powershell
New-Item -ItemType Directory -Force .\tmp_soutenance
.\cmake-build-debug\train_cli.exe `
  --model perceptron `
  --csv .\data\batiments_3_classes.csv `
  --output .\tmp_soutenance\perceptron.model `
  --report .\tmp_soutenance\perceptron.json `
  --seed 42
```

### Tester predict_cli

Le fichier `features.txt` doit contenir exactement 1 024 valeurs normalisées.

```powershell
.\cmake-build-debug\predict_cli.exe `
  .\models\buildings_3classes_32x32_mlp_v2.model `
  .\features.txt `
  --expected-class-count 3
```

### Lancer Flask

```powershell
.\.venv\Scripts\python.exe .\web\app.py
```

Ouvrir ensuite `http://127.0.0.1:5000`.

## 9. Limites du projet

- Les images 32 × 32 en niveaux de gris perdent beaucoup de détails.
- Le dataset reste petit pour un problème de reconnaissance d'images.
- Art déco et Art nouveau sont souvent confondus.
- Les scores des différents algorithmes ne sont pas directement comparables.
- Le SVM et le Perceptron utilisent des frontières linéaires.
- RBF v2 est corrigé, mais reste moins bon que MLP v2.

Le notebook de comparaison contient aussi de la régression. Notre projet final fait uniquement de la classification : il prédit une classe, pas une valeur continue. Il serait donc trompeur de présenter la régression comme une fonction de notre bibliothèque actuelle.

## 10. Conclusion

Le projet suit un chemin simple :

```text
images → CSV → entraînement C++ → modèle sauvegardé → prédiction C++ → Flask
```

Pour la soutenance, on peut montrer dans cet ordre :

1. les cas simples avec `demo.exe --tests` ;
2. un entraînement avec `train_cli.exe` ;
3. le modèle et le rapport sauvegardés ;
4. une prédiction avec `predict_cli.exe` ;
5. la même prédiction depuis l'interface Flask ;
6. les résultats et matrices de confusion dans ce notebook.

MLP v2 est le meilleur modèle à présenter dans l'interface. RBF v2 est utile pour expliquer une vraie démarche de diagnostic et de correction.